In [1]:
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

DARPA_DIR  = Path("../data/darpa/ta1-cadets-e3-official")
DARPA_DIR1 = Path("../data/darpa/ta1-cadets-e3-official-1")
DARPA_DIR2 = Path("../data/darpa/ta1-cadets-e3-official-2")
OUTPUT_DIR = Path("../data/processed")
EMB_DIR    = Path("../data/embeddings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED        = 42
BENIGN_SAMPLE_SIZE = 3000   # benign events to sample for balance

print("Paths set. Starting DARPA CADETS E3 preprocessing.")

Paths set. Starting DARPA CADETS E3 preprocessing.


In [2]:
# Ground truth: attacker-controlled IPs from the ground truth report section 3.1, 3.8, 3.13, 3.14
# These are the C2, shellcode, and payload delivery addresses documented for CADETS sessions.
# Any NetFlowObject connecting to/from these IPs during the attack windows is malicious.

ATTACKER_IPS = {
    # Session 1 (20180406) - Nginx Backdoor
    "81.49.200.166",    # http_post (exploit trigger)
    "78.205.235.65",    # shellcode_server
    "200.36.109.214",   # loaderDrakon delivery
    "139.123.0.113",    # drakon.freebsd.x64
    "152.111.159.139",  # libdrakon.freebsd.x64.so
    "154.143.113.18",   # netrecon (failed)
    "61.167.39.128",    # netrecon (success)
    # Session 2 (20180411) - Nginx re-exploit
    "25.159.96.207",    # http_post
    "76.56.184.25",     # shellcode_server
    "155.162.39.48",    # loaderDrakon
    "198.115.236.119",  # libdrakon
    # Session 3 (20180412) - Nginx + micro APT
    "53.158.101.118",   # drakon.freebsd.x64 (XIM process)
    "98.15.44.232",     # micro APT C2 (failed)
    "192.113.144.28",   # micro APT C2 (sendmail process)
    # Session 4 (20180413) - Final inject attempts
    # same IPs as session 2 and 3
    # Common Threat - phishing via postfix on CADETS
    "62.83.155.175",    # phishing email sender connecting to port 25
}

# Attack time windows in nanoseconds (Unix epoch * 1e9)
# Derived from event logs in the ground truth report.
# We add a 30-minute buffer on each side to capture setup/teardown activity.
import datetime, time

def et_to_ns(date_str, time_str):
    """Convert Eastern Time string to Unix nanoseconds.
    CADETS ran on the US East Coast (UTC-4 during April 2018 DST)."""
    dt = datetime.datetime.strptime(f"{date_str} {time_str}", "%Y-%m-%d %H:%M")
    # April 2018 is EDT = UTC-4
    dt_utc = dt + datetime.timedelta(hours=4)
    return int(dt_utc.timestamp() * 1e9)

BUFFER_NS = 30 * 60 * int(1e9)  # 30-minute buffer

ATTACK_WINDOWS = [
    # (start_ns, end_ns, primary_technique, description)
    (
        et_to_ns("2018-04-06", "10:51") - BUFFER_NS,
        et_to_ns("2018-04-06", "12:08") + BUFFER_NS,
        "T1190",
        "S1: Nginx exploit + loaderDrakon + netrecon + inject attempt"
    ),
    (
        et_to_ns("2018-04-06", "14:30") - BUFFER_NS,
        et_to_ns("2018-04-06", "15:30") + BUFFER_NS,
        "T1566",
        "S_phish: Phishing emails sent via CADETS postfix server"
    ),
    (
        et_to_ns("2018-04-11", "14:38") - BUFFER_NS,
        et_to_ns("2018-04-11", "15:15") + BUFFER_NS,
        "T1190",
        "S2: Nginx re-exploit + libdrakon + inject attempt"
    ),
    (
        et_to_ns("2018-04-12", "13:30") - BUFFER_NS,
        et_to_ns("2018-04-12", "14:38") + BUFFER_NS,
        "T1046",
        "S3: Nginx exploit + XIM drakon + micro APT port scans"
    ),
    (
        et_to_ns("2018-04-13", "08:34") - BUFFER_NS,
        et_to_ns("2018-04-13", "09:15") + BUFFER_NS,
        "T1055",
        "S4: Nginx exploit + pEja72mA + repeated inject attempts"
    ),
]

# Behaviour-to-technique mapping used when we have finer-grained event context
# These override the window-level primary technique when the event type matches.
BEHAVIOUR_TECHNIQUE_MAP = {
    "EVENT_CONNECT_attacker":     ("T1071.001", "Application Layer Protocol: Web Protocols"),
    "EVENT_EXECUTE_payload":      ("T1059",     "Command and Scripting Interpreter"),
    "EVENT_WRITE_payload":        ("T1105",     "Ingress Tool Transfer"),
    "EVENT_READ_sensitive":       ("T1083",     "File and Directory Discovery"),
    "inject_attempt":             ("T1055",     "Process Injection"),
    "port_scan":                  ("T1046",     "Network Service Discovery"),
    "nginx_exploit":              ("T1190",     "Exploit Public-Facing Application"),
    "phishing_email":             ("T1566",     "Phishing"),
}

# Known malicious process/file names from the ground truth report
MALICIOUS_NAMES = {
    "vugefal", "vugefai",         # drakon implant (session 1)
    "libdrakon", "devc",           # libdrakon module
    "grain",                       # libdrakon (session 2)
    "xim",                         # drakon process (session 3)
    "sendmail",                    # micro APT process disguised as sendmail
    "tmux-1002", "minions",        # micro APT staging names
    "font", "netlog",              # libdrakon staging
    "peja72ma",                    # drakon process (session 4)
    "ewq10bvcx", "memhelp.so",     # libdrakon (session 4)
    "eraseme", "done.so",          # inject staging
    "loaddrakon", "loaderdrakon",  # loader variants
    "drakon",                      # generic drakon references
}

print(f"Defined {len(ATTACKER_IPS)} known attacker IPs")
print(f"Defined {len(ATTACK_WINDOWS)} attack time windows")
print(f"Defined {len(MALICIOUS_NAMES)} known malicious process/file names")

Defined 15 known attacker IPs
Defined 5 attack time windows
Defined 19 known malicious process/file names


In [3]:
# CDM stream parser
#
# Each line of the CADETS JSON files is a CDM18 record wrapped in a datum envelope.
# We first do a single pass to build a UUID lookup table of all entities
# (subjects/processes, file objects, network flow objects).
# A second pass then resolves events using the lookup table.
#
# Files are split across segments:
#   ta1-cadets-e3-official/    -> initial segment (April 02 - April 06 crash)
#   ta1-cadets-e3-official-1/  -> after first crash (April 06 ~14:00 onwards)
#   ta1-cadets-e3-official-2/  -> after second crash (April 11 ~16:40 onwards)
# The operational event log confirms topic changes match these crash events.

CDM_PREFIX = "com.bbn.tc.schema.avro.cdm18."

def iter_records(dir_path: Path):
    """Yield parsed CDM datum objects from all .json files in a directory,
    in filename order. Each record is the inner typed object, e.g.
    {"uuid": ..., "type": ..., ...} with a synthetic key 'cdm_type'."""
    for fp in sorted(dir_path.glob("*.json*")):
        with open(fp, "r", encoding="utf-8", errors="replace") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    wrapper = json.loads(line)
                    datum   = wrapper.get("datum", {})
                    for full_key, obj in datum.items():
                        cdm_type = full_key.replace(CDM_PREFIX, "")
                        obj["cdm_type"] = cdm_type
                        yield obj
                except json.JSONDecodeError:
                    continue


def resolve_uuid(uuid_field):
    """Extract UUID string from the nested CDM UUID wrapper."""
    if isinstance(uuid_field, dict):
        return uuid_field.get(f"{CDM_PREFIX}UUID", uuid_field.get("UUID", ""))
    return str(uuid_field) if uuid_field else ""


def extract_properties(obj):
    """Flatten the properties.map dict from a CDM object."""
    props = obj.get("properties") or {}
    if isinstance(props, dict):
        return props.get("map", {})
    return {}


print("CDM parser functions defined.")

CDM parser functions defined.


In [4]:
# Pass 1: Build entity lookup tables
#
# We need to resolve UUIDs to human-readable descriptions when we encounter
# Events in Pass 2. Entities (Subjects, FileObjects, NetFlowObjects) always
# appear before the Events that reference them in the CDM stream.
#
# subjects    : uuid -> {pid, ppid, name, cmdLine}
# files       : uuid -> {path}
# netflows    : uuid -> {remoteAddr, remotePort, localPort}
# principals  : uuid -> {userId}

subjects   = {}
files      = {}
netflows   = {}
principals = {}

ALL_DIRS = [DARPA_DIR, DARPA_DIR1, DARPA_DIR2]

entity_count = 0
for data_dir in ALL_DIRS:
    for obj in iter_records(data_dir):
        ctype = obj.get("cdm_type", "")
        uuid  = resolve_uuid(obj.get("uuid"))
        if not uuid:
            continue

        if ctype == "Subject":
            props = extract_properties(obj)
            subjects[uuid] = {
                "pid":     obj.get("pid", ""),
                "ppid":    obj.get("ppid", ""),
                "name":    props.get("name", ""),
                "cmdLine": props.get("cmdLine") or obj.get("cmdLine", ""),
            }
            entity_count += 1

        elif ctype == "FileObject":
            props = extract_properties(obj)
            files[uuid] = {
                "path": obj.get("url") or props.get("path", ""),
            }
            entity_count += 1

        elif ctype == "NetFlowObject":
            netflows[uuid] = {
                "remoteAddr": obj.get("remoteAddress", ""),
                "remotePort": obj.get("remotePort",    ""),
                "localPort":  obj.get("localPort",     ""),
            }
            entity_count += 1

        elif ctype == "Principal":
            principals[uuid] = {
                "userId": obj.get("userId", ""),
            }
            entity_count += 1

print(f"Pass 1 complete: {entity_count} entities indexed")
print(f"  Subjects:   {len(subjects)}")
print(f"  Files:      {len(files)}")
print(f"  NetFlows:   {len(netflows)}")
print(f"  Principals: {len(principals)}")

Pass 1 complete: 2883416 entities indexed
  Subjects:   224629
  Files:      2305159
  NetFlows:   155322
  Principals: 22


In [5]:
# Labelling helpers
#
# An event is labelled as attack if ANY of the following is true:
#   1. Its timestamp falls within a documented attack window AND
#      it involves a known attacker IP or malicious process name
#   2. It involves a known attacker IP (regardless of timestamp)
#      — accounts for C2 beaconing outside the primary window
#
# The primary technique is assigned from the attack window that contains
# the event. If behaviour-specific signals are present (inject, scan, etc.)
# we refine the technique using BEHAVIOUR_TECHNIQUE_MAP.

def get_attack_window_technique(ts_ns):
    """Return (technique_id, description) if ts_ns falls in a known attack window."""
    for start, end, technique, desc in ATTACK_WINDOWS:
        if start <= ts_ns <= end:
            return technique, desc
    return None, None


def is_malicious_name(name: str) -> bool:
    """True if any known malicious name appears as a substring."""
    name_lower = name.lower()
    return any(m in name_lower for m in MALICIOUS_NAMES)


def classify_event(ts_ns, event_type, proc_name, file_path, remote_addr, cmdline):
    """Return (label, technique_id, tactic) for this event.
    label is 'ATTACK' or 'BENIGN'."""
    involves_attacker_ip = remote_addr in ATTACKER_IPS
    involves_malicious   = is_malicious_name(proc_name) or is_malicious_name(file_path)
    window_tech, _       = get_attack_window_technique(ts_ns)

    is_attack = involves_attacker_ip or (window_tech and involves_malicious)

    if not is_attack:
        return "BENIGN", "BENIGN", "Benign"

    # Assign technique based on available context
    tech_id = window_tech or "T1190"  # default to initial access if no window

    # Refine based on event-level signals
    if "inject" in cmdline.lower() or "inject" in proc_name.lower():
        tech_id = "T1055"
    elif event_type in ("EVENT_EXECUTE", "EVENT_CLONE", "EVENT_FORK") and involves_malicious:
        tech_id = "T1059"
    elif event_type in ("EVENT_WRITE", "EVENT_MMAP") and involves_malicious:
        tech_id = "T1105"
    elif event_type in ("EVENT_CONNECT", "EVENT_SENDTO", "EVENT_ACCEPT") and involves_attacker_ip:
        tech_id = "T1071.001"
    elif "scan" in cmdline.lower() or "scan" in proc_name.lower():
        tech_id = "T1046"
    elif "postfix" in proc_name.lower() or "sendmail" in file_path.lower():
        if remote_addr == "62.83.155.175":
            tech_id = "T1566"

    TACTIC_MAP = {
        "T1190":    "Initial Access",
        "T1566":    "Initial Access",
        "T1071.001":"Command And Control",
        "T1105":    "Command And Control",
        "T1059":    "Execution",
        "T1055":    "Defense Evasion",
        "T1046":    "Discovery",
        "T1083":    "Discovery",
    }
    tactic = TACTIC_MAP.get(tech_id, "Unknown")
    return "ATTACK", tech_id, tactic


print("Labelling functions defined.")

Labelling functions defined.


In [6]:
# Pass 2: Parse events and build alert_text strings
#
# For each Event record we:
#   1. Resolve subject UUID -> process name and command line
#   2. Resolve predicateObject UUID -> file path or remote IP:port
#   3. Build a natural-language alert_text string comparable to CIC-IDS format
#   4. Classify as ATTACK or BENIGN using the labelling function above
#
# Event types we keep (network and process events most relevant to SIEM triage):
KEEP_EVENT_TYPES = {
    "EVENT_EXECUTE", "EVENT_FORK", "EVENT_CLONE",
    "EVENT_CONNECT", "EVENT_ACCEPT", "EVENT_SENDTO", "EVENT_RECVFROM",
    "EVENT_READ", "EVENT_WRITE", "EVENT_OPEN", "EVENT_MMAP",
    "EVENT_RENAME", "EVENT_UNLINK",
}

PORT_NAMES = {
    21: "FTP", 22: "SSH", 25: "SMTP", 53: "DNS",
    80: "HTTP", 443: "HTTPS", 8000: "HTTP-alt", 8080: "HTTP-alt",
}


def build_alert_text(event_type, proc_name, cmdline, file_path, remote_addr, remote_port):
    """Build a short natural-language alert description.
    Format mirrors the CIC-IDS alert_text to keep downstream notebooks compatible."""
    proc   = proc_name or "unknown_process"
    action = event_type.replace("EVENT_", "").lower()

    if remote_addr:
        port_name = PORT_NAMES.get(int(remote_port) if remote_port else 0,
                                    f"port_{remote_port}")
        target = f"{remote_addr}:{remote_port} ({port_name})"
        return (f"Process {proc} {action} to {target}. "
                f"Cmd: {cmdline[:120] if cmdline else 'none'}.")
    elif file_path:
        return (f"Process {proc} {action} file {file_path}. "
                f"Cmd: {cmdline[:120] if cmdline else 'none'}.")
    else:
        return (f"Process {proc} {action}. "
                f"Cmd: {cmdline[:120] if cmdline else 'none'}.")


records = []
seen_event_ids = set()
skipped = 0

for data_dir in ALL_DIRS:
    for obj in iter_records(data_dir):
        if obj.get("cdm_type") != "Event":
            continue

        event_type = obj.get("type", "")
        if event_type not in KEEP_EVENT_TYPES:
            skipped += 1
            continue

        # Deduplicate by event UUID
        event_uuid = resolve_uuid(obj.get("uuid"))
        if event_uuid in seen_event_ids:
            skipped += 1
            continue
        seen_event_ids.add(event_uuid)

        ts_ns = obj.get("timestampNanos", 0) or 0

        # Resolve subject (process)
        subj_uuid  = resolve_uuid(obj.get("subject"))
        subj       = subjects.get(subj_uuid, {})
        proc_name  = subj.get("name", "")
        cmdline    = subj.get("cmdLine", "")
        props      = extract_properties(obj)
        if not cmdline:
            cmdline = props.get("cmdLine", "")

        # Resolve predicate object (file or network)
        pred_uuid   = resolve_uuid(obj.get("predicateObject"))
        file_path   = ""
        remote_addr = ""
        remote_port = ""

        if pred_uuid in files:
            file_path = files[pred_uuid].get("path", "")
        elif pred_uuid in netflows:
            nf          = netflows[pred_uuid]
            remote_addr = nf.get("remoteAddr", "")
            remote_port = str(nf.get("remotePort", ""))

        label, tech_id, tactic = classify_event(
            ts_ns, event_type, proc_name, file_path, remote_addr, cmdline
        )

        alert_text = build_alert_text(
            event_type, proc_name, cmdline, file_path, remote_addr, remote_port
        )

        records.append({
            "event_uuid":        event_uuid,
            "timestamp_ns":      ts_ns,
            "event_type":        event_type,
            "proc_name":         proc_name,
            "cmdline":           cmdline[:200],
            "file_path":         file_path,
            "remote_addr":       remote_addr,
            "remote_port":       remote_port,
            "label":             label,
            "attck_technique_id":tech_id,
            "attck_tactic":      tactic,
            "alert_text":        alert_text,
        })

df = pd.DataFrame(records)
print(f"Pass 2 complete: {len(df)} events parsed | {skipped} skipped")
print(f"\nLabel distribution:")
print(df["label"].value_counts().to_string())
print(f"\nAttack technique distribution:")
print(df[df["label"]=="ATTACK"]["attck_technique_id"].value_counts().to_string())

Pass 2 complete: 15815051 events parsed | 25535844 skipped

Label distribution:
label
BENIGN    15813821
ATTACK        1230

Attack technique distribution:
attck_technique_id
T1190        987
T1071.001    175
T1566         68


In [7]:
# Stratified sampling
#
# Keep ALL attack events (they are relatively few compared to benign).
# Sample BENIGN_SAMPLE_SIZE benign events to create a manageable, balanced dataset.
# This matches the approach used for CIC-IDS and is computationally necessary
# given the 42-million-event scale of the full CADETS dataset.

attack_df = df[df["label"] == "ATTACK"].copy()
benign_df = df[df["label"] == "BENIGN"].copy()

n_benign = min(len(benign_df), BENIGN_SAMPLE_SIZE)
sampled_benign = benign_df.sample(n=n_benign, random_state=RANDOM_SEED)

final_df = pd.concat([attack_df, sampled_benign], ignore_index=True)
final_df = final_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
final_df.index.name = "sample_id"
final_df["sample_id"] = final_df.index

print("FINAL DATASET STATISTICS")
print(f"Total rows:   {len(final_df)}")
print(f"Attack rows:  {len(attack_df)}")
print(f"Benign rows:  {n_benign}")
print("\nTactic distribution:")
print(final_df["attck_tactic"].value_counts().to_string())
print("\nTechnique distribution (attack only):")
print(final_df[final_df["label"]=="ATTACK"]["attck_technique_id"].value_counts().to_string())

# Preview alert_text samples
print("\nSample attack alert_text entries:")
for txt in final_df[final_df["label"]=="ATTACK"]["alert_text"].head(5).tolist():
    print(f"  {txt}")

FINAL DATASET STATISTICS
Total rows:   4230
Attack rows:  1230
Benign rows:  3000

Tactic distribution:
attck_tactic
Benign                 3000
Initial Access         1055
Command And Control     175

Technique distribution (attack only):
attck_technique_id
T1190        987
T1071.001    175
T1566         68

Sample attack alert_text entries:
  Process unknown_process recvfrom to 139.123.0.113:80 (HTTP). Cmd: none.
  Process unknown_process recvfrom to 76.56.184.25:80 (HTTP). Cmd: none.
  Process unknown_process sendto to 155.162.39.48:80 (HTTP). Cmd: none.
  Process unknown_process recvfrom to 139.123.0.113:80 (HTTP). Cmd: none.
  Process unknown_process sendto to 155.162.39.48:80 (HTTP). Cmd: none.


In [8]:
# Save processed dataset
# The output schema matches the CIC-IDS processed CSV so that notebooks 2-4
# can be pointed at either dataset without modification.

output_path = OUTPUT_DIR / "darpa_cadets_processed.csv"
final_df.to_csv(output_path)

# Also save to the embeddings directory under the expected filename
# so notebook 2 can load it with the same path it uses for CIC-IDS
emb_path = EMB_DIR / "darpa_cadets_sample.csv"
final_df.to_csv(emb_path)

print(f"Saved processed dataset to {output_path}")
print(f"Saved embeddings input to   {emb_path}")

Saved processed dataset to ../data/processed/darpa_cadets_processed.csv
Saved embeddings input to   ../data/embeddings/darpa_cadets_sample.csv


In [9]:
# Embedding
#
# Uses the same model (all-MiniLM-L6-v2) and parameters as the CIC-IDS pipeline
# to ensure the vector spaces are comparable across both datasets.

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
texts = final_df["alert_text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

assert len(embeddings) == len(final_df)
print(f"Shape: {embeddings.shape}")
print(f"Norm check: {np.linalg.norm(embeddings[0]):.4f}")

emb_npy = EMB_DIR / "darpa_cadets_embeddings.npy"
np.save(emb_npy, embeddings)

meta = {
    "dataset":     "DARPA TC E3 CADETS",
    "model":       "all-MiniLM-L6-v2",
    "device":      "cuda",
    "rows":        len(final_df),
    "dim":         384,
    "seed":        RANDOM_SEED,
    "batch_size":  64,
    "normalized":  True,
    "attack_rows": int((final_df["label"]=="ATTACK").sum()),
    "benign_rows": int((final_df["label"]=="BENIGN").sum()),
    "techniques":  final_df["attck_technique_id"].value_counts().to_dict(),
}
json_path = EMB_DIR / "darpa_cadets_embedding_meta.json"
json.dump(meta, open(json_path, "w"), indent=2)
print(f"\nEmbeddings saved to {emb_npy}")
print(f"Metadata saved to   {json_path}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/67 [00:00<?, ?it/s]

Shape: (4230, 384)
Norm check: 1.0000

Embeddings saved to ../data/embeddings/darpa_cadets_embeddings.npy
Metadata saved to   ../data/embeddings/darpa_cadets_embedding_meta.json
